# Featherweight — Phase 3 Group C: base Llama-3.1-8B BFCL baseline (Colab T4)

Runs the **base** model through the real BFCL harness under the *same*
`SalesforceLlamaHandler` prompting contract our fine-tune was trained on, so the
eventual base-vs-FT delta is apples-to-apples. GPT-4o was already done on the Mac.

**Approach (resolved from pinned bfcl-eval 2026.3.23 source):**
- We launch our **own** quantized vLLM server (bfcl's built-in launch can't pass
  `--quantization`), then point bfcl at it with `--skip-server-setup`.
- `scripts/run_bfcl.py` registers a custom `featherweight-base` entry
  (`SalesforceLlamaHandler`, `is_fc_model=False` => prompting path) then runs the
  bfcl CLI in-process.

> **Runtime risk:** bnb-4bit + vLLM on a T4 (Turing/SM 75) is the least-trodden
> path. If the server won't start, see the fallback note in the serve cell.


## 1. GPU + install

In [ ]:
!nvidia-smi

In [ ]:
# vLLM (server) + bfcl-eval (harness) + soundfile (dep bfcl doesn't pin).
# Pin transformers<5: bfcl's deps pull transformers 5.x which breaks vLLM's
# tokenizer (TokenizersBackend has no all_special_tokens_extended).
!pip install -q vllm 'bfcl-eval==2026.3.23' soundfile 'transformers<5' 'bitsandbytes>=0.46.1'  # bitsandbytes: vLLM bnb quantizer

## 2. Repo + package

In [ ]:
import os

REPO = "/content/Featherweight"
![ -d {REPO}/.git ] || git clone https://github.com/Nishaant-Soni/Featherweight {REPO}
%cd {REPO}
!pip install -q -e .
import sys

sys.path.insert(0, f"{REPO}/src")  # editable .pth is read only at startup
from featherweight import config

print("ROOT_DIR:", config.ROOT_DIR)
CATS = ",".join(config.CONFIG.eval.categories)
print("categories:", CATS)

## 3. BFCL project root + serving env
`BFCL_PROJECT_ROOT` is where bfcl writes `result/`+`score/` (kept under the repo).

In [3]:
import os

BASE_ID = config.BASE_MODEL_4BIT  # unsloth/llama-3.1-8b-Instruct-bnb-4bit (ungated)
os.environ["BFCL_PROJECT_ROOT"] = f"{REPO}/third_party/bfcl"
os.environ["LOCAL_SERVER_ENDPOINT"] = "localhost"
os.environ["LOCAL_SERVER_PORT"] = "8000"
os.makedirs(os.environ["BFCL_PROJECT_ROOT"], exist_ok=True)
print("BFCL_PROJECT_ROOT =", os.environ["BFCL_PROJECT_ROOT"])
print("serving base id   =", BASE_ID)

BFCL_PROJECT_ROOT = /content/Featherweight/third_party/bfcl
serving base id   = unsloth/llama-3.1-8b-Instruct-bnb-4bit


## 4. Launch our own quantized vLLM server (background)
Serves the bnb-4bit base under its real id (so bfcl's request `model` + tokenizer
match). `--dtype half` because the T4 has no bf16.

**Fallback if this fails:** drop `--load-format bitsandbytes` (for some vLLM
versions `--quantization bitsandbytes` alone suffices), lower `--max-model-len`,
or serve `meta-llama/Llama-3.1-8B-Instruct` with `--quantization bitsandbytes`
(needs an HF token; gated).

In [ ]:
!pkill -f "vllm serve" 2>/dev/null; sleep 3

import subprocess, time, requests

# log to a file so we can see progress/errors (subprocess stdout isn't shown in the cell)
logf = open("/content/vllm.log", "w")
server = subprocess.Popen(
    [
        "vllm",
        "serve",
        BASE_ID,
        "--quantization",
        "bitsandbytes",
        "--load-format",
        "bitsandbytes",
        "--dtype",
        "half",
        "--max-model-len",
        "8192",
        "--gpu-memory-utilization",
        "0.9",
        "--port",
        "8000",
    ],
    stdout=logf,
    stderr=subprocess.STDOUT,
)
print("launched pid", server.pid)
for i in range(120):
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).status_code == 200:
            print("vLLM server ready")
            break
    except Exception:
        pass
    print(f"--- waiting ({i * 5}s); last log lines: ---")
    !tail -n 4 /content/vllm.log
    time.sleep(5)
else:
    !tail -n 40 /content/vllm.log
    raise RuntimeError("vLLM server did not become ready - see log above")

## 5. Generate (base, via our own server)

Two speedups baked in here, because the **un-fine-tuned base rambles** toward the
token cap (slow) and bnb-4bit decode on a T4 isn't fast:
- a thin driver (`run_bfcl_fast.py`) that **clamps each completion to 512 tokens**
  (tool-call arrays are short; the tail tokens are runaway text we'd discard anyway);
- **`--num-threads 64`** so vLLM batches many requests at once.

`--allow-overwrite` redoes any earlier (failed) results.

In [5]:
%%writefile /content/run_bfcl_fast.py
# Register featherweight-base, then run the bfcl CLI with completions clamped to 512
# tokens (the base model otherwise generates toward the 4096 cap -> very slow).
from featherweight.eval import bfcl_register
bfcl_register.register_base_model()

from openai.resources.completions import Completions
_orig = Completions.create
def _capped(self, *args, **kwargs):
    mt = kwargs.get('max_tokens')
    if mt is None or mt > 512:
        kwargs['max_tokens'] = 512
    return _orig(self, *args, **kwargs)
Completions.create = _capped

from bfcl_eval.__main__ import cli
cli()

Writing /content/run_bfcl_fast.py


In [6]:
!python /content/run_bfcl_fast.py generate --model featherweight-base \
    --backend vllm --skip-server-setup --num-threads 32 \
    --test-category {CATS} --allow-overwrite

Generating results for ['featherweight-base']
Running full test cases for categories: ['irrelevance', 'multiple', 'parallel', 'parallel_multiple', 'simple_python'].
Max context length: 131072
server is ready!
Generating results for featherweight-base: 100% 1240/1240 [2:26:58<00:00,  7.11s/it] 


## 6. Evaluate (CPU scoring)

In [8]:
!python scripts/run_bfcl.py evaluate --model featherweight-base --test-category {CATS}

Number of models evaluated:   0% 0/1 [00:00<?, ?it/s]🦍 Model: featherweight-base
🔍 Running test: multiple
✅ Test completed: multiple. 🎯 Accuracy: 31.00%
🔍 Running test: parallel
✅ Test completed: parallel. 🎯 Accuracy: 33.50%
🔍 Running test: irrelevance
✅ Test completed: irrelevance. 🎯 Accuracy: 90.42%
🔍 Running test: simple_python
✅ Test completed: simple_python. 🎯 Accuracy: 36.50%
🔍 Running test: parallel_multiple
✅ Test completed: parallel_multiple. 🎯 Accuracy: 21.50%
Number of models evaluated: 100% 1/1 [00:00<00:00,  1.63it/s]
📈 Aggregating data to generate leaderboard score table...
🏁 Evaluation completed. See /content/Featherweight/third_party/bfcl/score/data_overall.csv for overall evaluation results on BFCL V4.
See /content/Featherweight/third_party/bfcl/score/data_live.csv, /content/Featherweight/third_party/bfcl/score/data_non_live.csv, /content/Featherweight/third_party/bfcl/score/data_multi_turn.csv, /content/Featherweight/third_party/bfcl/score/data_agentic.csv and /conten

## 7. Parse scores + stop the server
Copy the score dir out (download / Drive / HF) so the local report can combine it
with the GPT-4o baseline via `report.write_baselines`.

In [9]:
from pathlib import Path
from featherweight.eval import report

score_dir = Path(os.environ["BFCL_PROJECT_ROOT"]) / "score" / "featherweight-base"
for cat, s in report.collect_scores(score_dir).items():
    print(cat, s)

# zip the score dir for download
import shutil

shutil.make_archive("/content/base_scores", "zip", score_dir)
print("scores zipped -> /content/base_scores.zip")
server.terminate()

irrelevance {'accuracy': 0.9041666666666667, 'correct_count': 217, 'total_count': 240, 'invalid_rate': 0.0}
multiple {'accuracy': 0.31, 'correct_count': 62, 'total_count': 200, 'invalid_rate': 0.205}
parallel_multiple {'accuracy': 0.215, 'correct_count': 43, 'total_count': 200, 'invalid_rate': 0.17}
parallel {'accuracy': 0.335, 'correct_count': 67, 'total_count': 200, 'invalid_rate': 0.05}
simple_python {'accuracy': 0.365, 'correct_count': 146, 'total_count': 400, 'invalid_rate': 0.0775}
scores zipped -> /content/base_scores.zip
